In [ ]:
import json
import os
import sys
import urllib.request
import zipfile
from pathlib import Path
from time import perf_counter



os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
import time
from pyspark.sql import SparkSession


def stop_spark():
    global spark
    spark.catalog.clearCache()
    spark.stop()
    spark = None
    time.sleep(2)


def create_spark_session(num_cores):
    spark = (
        SparkSession.builder
        .master(f"local[{num_cores}]")
        .appName(f"als_ml1m_{num_cores}_cores")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark


def create_als():
    return ALS(
        userCol="user_id",
        itemCol="item_id",
        ratingCol="rating",
        rank=5,
        regParam=0.1,
        maxIter=10,
        numUserBlocks=4,
        numItemBlocks=4,
        coldStartStrategy="drop",
        seed=42,
    )

In [ ]:
core_results = []
CORE_VALUES = [1, 2, 4]
NUM_RUNS = 3
for num_cores in CORE_VALUES:
    print(f"\n{num_cores} core(s)")

    stop_spark()
    spark = create_spark_session(num_cores)

    print(
        "Default parallelism:",
        spark.sparkContext.defaultParallelism,
    )

    train_df = (
        spark.read
        .parquet("/home/anna/projects/ccdpp-pyspark-movielens/data/processed/ml-1m/train")
        .cache()
    )

    test_df = (
        spark.read
        .parquet("/home/anna/projects/ccdpp-pyspark-movielens/data/processed/ml-1m/test_clean")
        .cache()
    )


    train_count = train_df.count()
    test_count = test_df.count()

    print("Train rows:", train_count)
    print("Test rows:", test_count)

    als = create_als()

    evaluator = RegressionEvaluator(
        labelCol="rating",
        predictionCol="prediction",
        metricName="rmse",
    )


    _ = als.fit(train_df)

    for run in range(1, NUM_RUNS + 1):
        print(f"run {run}/{NUM_RUNS}")

        start = perf_counter()
        model = als.fit(train_df)
        training_time = perf_counter() - start

        predictions = (
            model.transform(test_df)
            .cache()
        )

        prediction_count = predictions.count()
        rmse = evaluator.evaluate(predictions)
        coverage = prediction_count / test_count

        core_results.append(
            {
                "dataset": "ml-1m",
                "algorithm": "ALS",
                "num_cores": num_cores,
                "run": run,
                "train_count": train_count,
                "test_count": test_count,
                "prediction_count": prediction_count,
                "training_time": training_time,
                "rmse": rmse,
                "coverage": coverage,
                "rank": 5,
                "reg_param": 0.1,
                "max_iter": 10,
                "num_user_blocks": 4,
                "num_item_blocks": 4,
            }
        )

        print(
            f"time={training_time:.3f}s, "
            f"RMSE={rmse:.6f}, "
            f"coverage={coverage:.6f}"
        )

        predictions.unpersist()

    train_df.unpersist()
    test_df.unpersist()

In [ ]:
core_results_df = spark.createDataFrame(core_results)

core_results_df.select(
    "num_cores",
    "run",
    "training_time",
    "rmse",
    "coverage",
).orderBy(
    "num_cores",
    "run",
).show(truncate=False)

In [ ]:
core_summary = (
    core_results_df
    .groupBy("num_cores")
    .agg(
        F.avg("training_time").alias("mean_time"),
        F.stddev("training_time").alias("std_time"),
        F.avg("rmse").alias("mean_rmse"),
        F.stddev("rmse").alias("std_rmse"),
        F.avg("coverage").alias("mean_coverage"),
        F.count("*").alias("num_runs"),
    )
)

In [ ]:
baseline_time = (
    core_summary
    .filter(F.col("num_cores") == 1)
    .select("mean_time")
    .first()[0]
)

In [ ]:
core_summary = (
    core_summary
    .withColumn(
        "speedup",
        F.lit(baseline_time) / F.col("mean_time"),
    )
    .withColumn(
        "efficiency",
        F.col("speedup") / F.col("num_cores"),
    )
    .orderBy("num_cores")
)

core_summary.show(truncate=False)

In [ ]:
(
    core_results_df
    .write
    .mode("overwrite")
    .parquet("results/ml-1m/als_core_scaling_runs")
)

In [ ]:
(
    core_summary
    .write
    .mode("overwrite")
    .parquet("results/ml-1m/als_core_scaling_summary")
)

In [ ]:
(
    core_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("results/ml-1m/als_core_scaling_summary_csv")
)

In [ ]:
spark.stop()